# 🏗️ Notebook 1: Discord — Requirements & Architecture


## 🛠️ Setup

```bash
cd 06-system-designs/discord
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.

All code in this lab is **self-contained Python** — no servers, no databases. We simulate Discord's gateway, pub/sub, and fan-out in memory so you can run every cell and see what happens.


## 💡 What we're designing

Discord is a **real-time chat** platform. Users join **guilds** (servers), talk in **text channels**, hop into **voice channels**, and see who is **online right now**.

Three things make Discord hard:

1. **Real-time**: a message typed in Tokyo should appear in New York in <200 ms.
2. **Huge fan-out**: a popular channel can have 100,000+ members watching live.
3. **Presence**: millions of online/offline flips per second without melting the servers.

### Functional requirements
- Send and receive text messages in a channel.
- See who is online (presence).
- Join voice channels with low latency.
- Keep message history (scrollback).
- Push notifications when a user is offline.

### Non-functional requirements
- **WebSockets**, not HTTP polling — pushing is cheaper than clients asking "any news?" 50 times per minute.
- **Horizontally scalable** gateway — one box can't hold 15 million sockets.
- **Geographically close** voice servers — audio latency is dominated by distance.


## 📏 Back-of-envelope capacity

Before drawing boxes, let's put numbers on the problem. Being wrong by 10× is fine; being wrong by 1000× means your architecture collapses.


In [ ]:
# ==== Capacity estimation ====
# Public-ish estimates. The point is the *method* -- and the sanity check at the end,
# which is the step people skip and the reason estimates end up 100x wrong.

MAU                   = 150_000_000    # monthly active users
concurrent_ratio      = 0.10           # ~10% of MAU online at peak
chatty_fraction       = 0.10           # of those online, how many are actually TYPING?
                                       # Most people sit in voice or lurk. Skipping this
                                       # factor is the classic way to overestimate by 10x.
msgs_per_chatter_min  = 2
avg_channel_members   = 50             # average audience for one message
avg_msg_bytes         = 250            # content + author + ids + timestamps
retention_years       = 5

concurrent_users = int(MAU * concurrent_ratio)
chatters         = int(concurrent_users * chatty_fraction)
msgs_per_sec     = chatters * msgs_per_chatter_min / 60
fanout_per_sec   = msgs_per_sec * avg_channel_members

# ---- Sanity check BEFORE trusting anything downstream ----
msgs_per_day = msgs_per_sec * 86_400

# ---- Storage: writes are cheap per message, expensive in aggregate ----
storage_gb_per_day = msgs_per_sec * avg_msg_bytes * 86_400 / 1e9
storage_pb_total   = storage_gb_per_day * 365 * retention_years / 1e6

# ---- Fan-out egress: the number that actually sizes the gateway fleet ----
fanout_egress_gbps = fanout_per_sec * avg_msg_bytes * 8 / 1e9

# ---- Voice ----
voice_bitrate_kbps = 64                # Opus at decent quality
concurrent_voice   = 1_000_000
avg_voice_channel  = 5                 # listeners per speaker in a typical channel
voice_ingress_gbps = concurrent_voice * voice_bitrate_kbps / 1e6
# An SFU forwards each speaker's stream to everyone else in the channel, so egress
# is roughly (members - 1) x ingress. Voice egress dwarfs text by orders of magnitude.
voice_egress_gbps  = voice_ingress_gbps * (avg_voice_channel - 1)

print(f'Concurrent users      : {concurrent_users:>15,}')
print(f'  of whom chatting    : {chatters:>15,}')
print(f'Messages / sec        : {msgs_per_sec:>15,.0f}')
print(f'Messages / day        : {msgs_per_day:>15,.0f}   <-- sanity check this against reality')
print(f'Fan-out events / sec  : {fanout_per_sec:>15,.0f}')
print()
print(f'Text storage / day    : {storage_gb_per_day:>14,.0f} GB')
print(f'Text storage @ {retention_years}y     : {storage_pb_total:>14,.1f} PB')
print(f'Text fan-out egress   : {fanout_egress_gbps:>14,.1f} Gbps')
print()
print(f'Voice ingress         : {voice_ingress_gbps:>14,.0f} Gbps')
print(f'Voice SFU egress      : {voice_egress_gbps:>14,.0f} Gbps  <-- {voice_egress_gbps/fanout_egress_gbps:.0f}x the text egress')

# Takeaway: ~2.5M fan-out events/sec is the number that shapes the text design.
# A single box can't do that -- we need fan-out via a message bus.
# And voice bandwidth is a completely separate, much larger problem: it is why
# voice does not go anywhere near the text gateway.

### Sanity-check the sanity check

Around **4 billion messages/day** is the number to argue about. If you had skipped
`chatty_fraction` and assumed all 15M concurrent users type 2 messages/minute, you'd have
gotten **43 billion/day** — an order of magnitude too big, and you'd have designed (and
budgeted) a system 10× larger than the one you need.

That is what back-of-envelope is *for*: not precision, but catching the assumption that is
wrong by 10×. Always write down the number a human can check ("messages per day") and not
just the number the architecture needs ("events per second").

⚖️ **What these numbers are still hiding:**

- `avg_channel_members = 50` is an *average* over a distribution with a brutal tail. Most
  channels have 5 members; a few have 500,000. The average sizes your fleet; the tail
  decides your architecture. Notebook 3 runs that case.
- Voice egress assumes a small channel. A 1-speaker / 10,000-listener stage channel is a
  different product with different economics (that's where you start transcoding and
  building a relay tree instead of a flat SFU).
- Storage ignores attachments, which in practice dwarf message text and go to a CDN, not
  to the message store.

## 🧱 High-level architecture

```
           ┌────────────┐                             
           │  Clients   │  (web / mobile / desktop)   
           └──────┬─────┘                             
                  │ wss://gateway.discord...          
                  ▼                                   
           ┌────────────┐     ┌──────────────┐        
           │  Gateway   │◀───▶│  Session &   │        
           │ (WebSocket)│     │  Presence    │        
           └──────┬─────┘     └──────────────┘        
                  │ publish                           
                  ▼                                   
           ┌─────────────────────┐                    
           │  Message Bus        │  Kafka / NATS      
           │  topic per channel  │                    
           └──────┬──────────────┘                    
                  │ subscribe                         
        ┌─────────┼─────────┐                         
        ▼         ▼         ▼                         
   ┌────────┐┌────────┐┌────────┐                     
   │Gateway ││Gateway ││Gateway │ … 300 instances     
   └────────┘└────────┘└────────┘                     
                  │                                   
                  ▼                                   
        ┌───────────────────┐     ┌──────────────┐    
        │ Message storage   │     │ Push notif   │    
        │ (Cassandra/Scylla)│     │ (APNS/FCM)   │    
        └───────────────────┘     └──────────────┘    

   Voice: separate UDP/WebRTC media servers (SFU) in each region.
```

Key ideas, in plain English:

- Every client keeps **one WebSocket open** to a Gateway. No polling.
- When Alice sends a message, the Gateway publishes it to a **channel topic** on the bus.
- Every Gateway that has *at least one subscriber* of that channel receives the event, and pushes to only those local sockets.
- Messages are persisted separately (for scrollback). The hot path is **write-then-publish**, not write-then-read.


## ⚖️ Gateway sharding — bad → best

A single WebSocket server comfortably holds ~50k connections (file descriptors, memory for buffers). With 15M concurrent users we need ~300 gateways. How do we pick which one a client connects to?

We'll simulate three approaches and watch the load distribution.


In [ ]:
# ==== Gateway assignment strategies ====
import random, hashlib, bisect
from collections import Counter

NUM_USERS = 300_000
NUM_GATEWAYS = 30
random.seed(7)
user_ids = [random.randint(10**18, 10**19) for _ in range(NUM_USERS)]

def skew(counter, n_bins):
    vals = [counter.get(i, 0) for i in range(n_bins)]
    avg = sum(vals) / n_bins
    return min(vals), max(vals), max(vals) / avg

def h(s) -> int:
    return int(hashlib.md5(str(s).encode()).hexdigest(), 16)

# ---------- ❌ BAD: random assignment ----------
# Even load, but a client that reconnects lands somewhere new every time, so its
# presence state and subscriptions have to be rebuilt from scratch. Cache always cold.
bad = Counter(random.randrange(NUM_GATEWAYS) for _ in user_ids)

# ---------- 🙂 OK: user_id % N ----------
# Sticky per user and trivially cheap. The catch shows up only when N changes.
def modulo_pick(uid, n): return uid % n
ok = Counter(modulo_pick(uid, NUM_GATEWAYS) for uid in user_ids)

# ---------- ✅ BEST: consistent hashing ----------
# Hash each gateway to VIRTUAL_NODES points on a ring; a user lands on the next
# point clockwise. Adding a gateway steals a slice from each neighbour instead of
# renumbering everybody.
VIRTUAL_NODES = 100          # more virtual nodes -> smoother distribution

def build_ring(n_gateways, vnodes=VIRTUAL_NODES):
    ring = sorted((h(f'gw{g}-{v}'), g) for g in range(n_gateways) for v in range(vnodes))
    return [p for p, _ in ring], [g for _, g in ring]

points, owners = build_ring(NUM_GATEWAYS)

def ring_pick(uid, points, owners):
    # Binary search for the first ring point >= hash(uid); wrap around at the end.
    i = bisect.bisect_left(points, h(uid))
    return owners[i % len(owners)]

best = Counter(ring_pick(uid, points, owners) for uid in user_ids)

print("Load distribution with 30 gateways (nothing has changed yet):")
for name, c in [('BAD  random', bad), ('OK   modulo', ok), ('BEST consistent', best)]:
    lo, hi, sk = skew(c, NUM_GATEWAYS)
    print(f'  {name:<16} min={lo:>5}  max={hi:>5}  max/avg skew={sk:.2f}')

In [ ]:
# ==== The part that actually matters: what happens on a deploy? ====
# Add ONE gateway (30 -> 31) and count how many users have to move.
# A moved user = a dropped WebSocket, a re-IDENTIFY, and a cold presence cache.

N2 = NUM_GATEWAYS + 1
points2, owners2 = build_ring(N2)

moved_modulo = sum(1 for uid in user_ids
                   if modulo_pick(uid, NUM_GATEWAYS) != modulo_pick(uid, N2))
moved_ring   = sum(1 for uid in user_ids
                   if ring_pick(uid, points, owners) != ring_pick(uid, points2, owners2))

print(f"Adding gateway #{N2} to a {NUM_GATEWAYS}-gateway fleet:\n")
print(f"  modulo      : {moved_modulo:>7,} of {NUM_USERS:,} users move  ({moved_modulo/NUM_USERS:.1%})")
print(f"  consistent  : {moved_ring:>7,} of {NUM_USERS:,} users move  ({moved_ring/NUM_USERS:.1%})")
print(f"  theory says : {1/N2:.1%} for consistent hashing (1/N)")
print()
print(f"Scaled to 15M concurrent users, that is {moved_modulo/NUM_USERS*15e6:,.0f} reconnects")
print(f"vs {moved_ring/NUM_USERS*15e6:,.0f}. The first one is a self-inflicted DDoS: every")
print("one of those clients simultaneously re-IDENTIFYs and re-downloads its guild list.")

**What to notice**

- On the steady-state load table, consistent hashing is the **worst** of the three
  (skew ~1.19 vs ~1.02): 3,000 random ring points do not carve a circle into 30 equal
  slices. So if balance were the only question, you would not pick it. Balance is not the
  question — a deploy is.
- Modulo moves ~97% of users when the fleet grows by one node. Consistent hashing moves ~3%,
  matching the 1/N theory. At Discord's scale that is the difference between a routine
  scale-up and a thundering herd of reconnects that takes the fleet down.
- Discord actually hashes by `guild_id`, not `user_id`, so all members of one server land on
  the same gateway and presence for that guild stays local — same trick, different key.

⚖️ **What consistent hashing costs you:**

- **Virtual nodes are a tuning knob you now own.** Drop `VIRTUAL_NODES` to 1 and re-run: the
  skew gets ugly, because 30 random points do not divide a ring evenly. 100–200 per node is
  the usual answer, and it costs memory and ring-rebuild time.
- **It balances *keys*, not *load*.** One guild with 500k members hashes to exactly one
  gateway. Consistent hashing will cheerfully melt that node while its neighbours idle —
  hot keys need a separate escape hatch (split the guild across gateways, or give it a
  dedicated fleet).
- **Every node needs the same ring.** That is a small piece of strongly-consistent cluster
  state (etcd/ZooKeeper) that must be updated atomically, or two gateways disagree about
  who owns a user and the client flaps between them.

## ✅ Summary

- Discord is **push, not pull** — WebSockets are non-negotiable at this scale.
- The dominant cost is **fan-out**, not storage.
- Gateways are **stateless** (session in Redis); we can scale them horizontally.
- Use **consistent hashing** to route clients so deploys don't cause a reconnect storm.

Next: [Notebook 2 — Data Model & APIs](./02_data_and_api.ipynb).
